# 산학프로젝트
**제조 공정 품질 불량 예측**

# 09_Preprocessing_Validation

- 02에서 정한 기록 오류 보정이 성능에 영향을 주는지 사후 확인
- 대상: 오류 3 `Average_Screw_RPM` 10배 기록 스케일
- 02는 보정을 **정하는** 곳이고, 성능 영향은 모델과 평가 프로토콜이 확정된 뒤라야 잴 수 있어 08 뒤에 둠
- 보정 여부의 판단 근거는 단위(mm/s)와 `Average ≤ Max` 물리 관계이며, 이 노트북 결과가 그 결정을 바꾸지 않음
- Output: `data/09_screw_rpm_treatment.csv`

## import

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import (StratifiedGroupKFold, StratifiedKFold,
                                     cross_val_predict, cross_val_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 경로설정

In [2]:
if Path.cwd().name!='data':
    os.chdir('data')

## 데이터 불러오기

- 보정 **전** 값이 필요하므로 이 노트북만 `labeled_data.csv`를 다시 읽음. 03 ~ 08은 `labeled_modeling.csv`만 사용
- 02와 같은 순서(중복 제거 → CN7·RG3 한정)로 원본을 재구성한 뒤 `labeled_modeling.csv` 인덱스에 맞춰 행을 일치시킴

In [3]:
# 02_EDA에서 저장한 전처리 데이터 (오류 3 보정이 적용된 상태)
model_df=pd.read_csv('labeled_modeling.csv', index_col=0)

# 보정 전 값 확보 - 02와 같은 순서로 원본 재구성
raw=pd.read_csv('labeled_data.csv').drop_duplicates(keep='first', ignore_index=True)
raw_cn7rg3=raw[raw['PART_NAME'].str.startswith(('CN7', 'RG3'))]

# 행이 어긋나면 비교 자체가 무의미하므로 인덱스 일치를 확인
assert raw_cn7rg3.index.equals(model_df.index), '원본과 모델링 데이터의 행이 어긋남'
raw_rpm=raw_cn7rg3['Average_Screw_RPM']

print(f'{model_df.shape[0]}행 {model_df.shape[1]}컬럼 / 불량 {int(model_df["PassOrFail"].sum())}건')
print('보정 대상(원본 100 초과) 행:', int((raw_rpm>100).sum()))
print(f'원본 범위 {raw_rpm.min():.2f} ~ {raw_rpm.max():.2f}'
      f' / 보정 후 {model_df["Average_Screw_RPM"].min():.2f} ~ {model_df["Average_Screw_RPM"].max():.2f}')

5230행 30컬럼 / 불량 60건
보정 대상(원본 100 초과) 행: 3021
원본 범위 29.00 ~ 293.90 / 보정 후 29.00 ~ 29.40


## 1. 세 가지 처리 비교

- 오류 3의 처리 방식 세 가지를 같은 모델로 비교
- **A. 원본**(보정 안 함) / **B. 변수 제외** / **C. 보정**(100 초과를 10으로 나눔)
- 무작위 5-fold와 생산일 그룹 5-fold 두 프로토콜에서 확인
- 이 변수를 빼도 생산일을 식별할 수 있는지도 함께 확인

In [4]:
rpm_base=model_df.drop(columns=['TimeStamp', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL',
                                'PART_NAME', 'EQUIP_CD'])
rpm_y=rpm_base['PassOrFail']
rpm_x=pd.get_dummies(rpm_base.drop(columns='PassOrFail'), columns=['Part'],
                     drop_first=True, dtype=int)

# A. 원본 = 보정 전 값 (raw_rpm은 위에서 원본에서 가져옴)
rpm_x_raw=rpm_x.copy()
rpm_x_raw['Average_Screw_RPM']=raw_rpm.to_numpy()

rpm_variants={
    'A. 원본': rpm_x_raw,
    'B. 변수 제외': rpm_x.drop(columns='Average_Screw_RPM'),
    'C. 100 초과를 10으로 나누기': rpm_x,
}

rpm_model=Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=20, min_samples_leaf=4,
                                  class_weight='balanced', n_jobs=-1, random_state=0)),
])
rpm_date=pd.to_datetime(model_df['TimeStamp']).dt.date.astype(str).to_numpy()
date_counts=pd.Series(rpm_date).value_counts()
date_keep=pd.Series(rpm_date).isin(date_counts[date_counts>=3].index).to_numpy()

rpm_rows=[]
for name, features in rpm_variants.items():
    random_proba=cross_val_predict(clone(rpm_model), features, rpm_y,
                                   cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=0),
                                   method='predict_proba', n_jobs=-1)[:, 1]
    group_proba=cross_val_predict(clone(rpm_model), features, rpm_y,
                                  cv=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0),
                                  groups=rpm_date, method='predict_proba', n_jobs=-1)[:, 1]

    cut=int(len(rpm_y)*0.1)
    top_idx=np.argsort(random_proba)[::-1][:cut]
    recall_at_10=rpm_y.to_numpy()[top_idx].sum()/rpm_y.sum()

    date_accuracy=cross_val_score(
        clone(Pipeline(steps=[('scaler', StandardScaler()),
                              ('rf', RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0))])),
        features[date_keep], rpm_date[date_keep],
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=0),
        scoring='accuracy', n_jobs=-1).mean()

    rpm_rows.append({
        '처리': name,
        '특성 수': features.shape[1],
        '무작위 AP': round(average_precision_score(rpm_y, random_proba), 4),
        '생산일 그룹 AP': round(average_precision_score(rpm_y, group_proba), 4),
        '무작위 R@10%': round(float(recall_at_10), 3),
        '생산일 분류 정확도': round(float(date_accuracy), 4),
    })

rpm_compare_df=pd.DataFrame(rpm_rows)
rpm_compare_df.to_csv('09_screw_rpm_treatment.csv', index=False, encoding='utf-8-sig')
rpm_compare_df

,처리,특성 수,무작위 AP,생산일 그룹 AP,무작위 R@10%,생산일 분류 정확도
0,A. 원본,26,0.3898,0.0190,0.783,0.9989
1,B. 변수 제외,25,0.3901,0.0234,0.783,0.9985
2,C. 100 초과를 10으로 나누기,26,0.3910,0.0198,0.783,0.9987


**세 처리 사이에 성능 차이가 없음**

- 이 변수를 빼도 **생산일 분류 정확도가 그대로**임. 구간 지문은 한 변수의 성질이 아니라 **변수 조합 전체의 성질**임 (`04_1_1_Split_Structure` 1절)
- 트리 모델은 **값의 순서만** 쓰므로 나누기든 곱하기든 결과가 같음. 따라서 **성능만 보고 보정 방향을 판단할 수 없고**, 단위와 물리 관계로 판단해야 함
- 차이는 불량 1 ~ 2건 수준이라 어느 처리도 이득이라 보기 어려움

**결론: 성능 목적으로는 이득이 없으나 C(보정)를 적용함**
- 학습과 서빙의 전처리를 일치시켜야 함
- unlabeled가 **전 구간 잘못된 스케일**이므로, 보정 없이 서빙하면 학습 분포와 10배 어긋난 값이 입력됨

## 정리

| 처리 | 성능 | 채택 |
| --- | --- | --- |
| A. 원본 | 기준 | 미채택 |
| B. 변수 제외 | 차이 없음 | 미채택 |
| C. 100 초과를 10으로 나눔 | 차이 없음 | **채택** |

- **성능은 보정 여부의 판단 근거가 되지 못함** — 트리 모델은 값의 순서만 쓰므로 전 구간을 같은 방향으로 10배 해도 분할 지점이 그대로임
- 그럼에도 C를 적용하는 이유는 **학습과 서빙의 전처리를 일치시키기 위해서**임. unlabeled가 전 구간 잘못된 스케일이므로 보정 없이 서빙하면 학습 분포와 10배 어긋난 값이 들어감
- 보정 규칙 자체와 판단 근거는 `02_EDA` 오류 3 절, 서빙 측 구현은 `dashboard/backend/app/corrections.py`